In [ ]:
%%capture
!pip install pgmpy

# Exemplos

In [ ]:
from mre import listar_exemplos, get_example_model

- **listar_exemplos**: lista as redes de exemplo do artigo disponíveis
  - circuito
  - viagem
  - academe
- **get_example_model**: tráz uma rede de exemplo do artigo

Listando exemplos disponíveis

In [ ]:
listar_exemplos()

Redes de exemplo disponíveis em mre_lib:

  'circuito'  — Circuito elétrico (Poole & Provan, 1991)
                4 gates (A,B,C,D), evidência: corrente detectada
                Reproduz Tabela 2 e 3 do paper

  'viagem'    — Férias do Sr. Smith (Shimony, 1993)
                n_trilhas=1 (default) ou n_trilhas=N (multi-estado)
                Reproduz Tabelas 4 e 5 do paper

  'academe'   — Desempenho acadêmico (Flores et al., 2005)
                4 alvos: Teoria, Prática, Extra, OutrosFatores
                Reproduz Tabela 6 do paper

Uso: modelo = get_example_model('circuito')


Obter um modelo pronto

In [ ]:
circuito = get_example_model('circuito')

/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in a future release. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Trazendo suas propriedades

In [ ]:
infer_circuito = circuito['infer']
alvos = circuito['alvos']
evidencia_padrao = circuito['evidencias']['corrente']
priors = circuito['priors']

Exibindo-as

In [ ]:
print("Alvos:", alvos)
print("Evidência:", evidencia_padrao)
print("Priors de falha (D):", priors['D'])

Alvos: ['A', 'B', 'C', 'D']
Evidência: {'Input': 'current', 'TotalOutput': 'current'}
Priors de falha (D): {'defective': 0.1, 'ok': 0.9}


# Métricas fundamentais

In [ ]:
from mre import gbf, bur, jeffreys

- **gbf**: Generalized Bayes Factor. Mede o quanto a evidência $e$ muda as chances da hipótese $x$
  - Parâmetros:
    - post: probabilidade posterior de $x$ dado a evidência $e$
    - prior: porbabilidade a priori de $x$
  - Retorna:
    - float: GBF
- **bur**: Belief Updating Ratio. Equivalente ao likelihood.
  - Parâmetros:
    - post
    - prior
  - Retorna:
    - float
- **jeffreys**: Classifica o valor de um GBF ou CBF a partir da escala de Jeffreys
  - Parâmetros:
    - GBF ou CBF

In [ ]:
# prior
prior_D = priors['D']['defective']
print(f'Prior: {prior_D}')

# posterior
q_D = infer_circuito.query(['D'],
                           evidence=evidencia_padrao,
                           show_progress=False)
post_D = q_D.get_value(D='defective')
print(f'Posterior: {post_D}')

# gbf
valor_gbf = gbf(post_D, prior_D)
print(f'GBF: {valor_gbf}')

# bur
valor_bur = bur(post_D, prior_D)
print(f'BUR: {valor_bur}')

forca = jeffreys(valor_gbf)
print(f'Força: {forca}')

Prior: 0.1
Posterior: 0.30112187047184324
GBF: 3.8777817186471326
BUR: 3.0112187047184324
Força: Substancial


# Panorama e Explaining-Away

In [ ]:
from mre import posteriors, gbf, cbf_condicional

- **posteriors**: Calcula os posteriors marginais de cada variável alvo e devolve um dicionário com todas as distribuições a posteriori marginalizadas
  - Parâmetros:
    - infer: VariableElimination do pgmpy
    - alvos: lista de variáveis alvo
    - evidencia: dicionário de evidências
  - Retorna:
    - dicionário com DiscreteFactor (pgmpy) de cada variável

- **cbf_condicional**: Calcula o CBF de uma variável condicionada em uma explicação, medindo o quanto $y$ acrescenta a uma explicação que já tem $x$.
  - Parâmetros:
    - infer: VariableElimination do pgmpy
    - alvo_spec: dicionário variável sendo avaliada com seu estado
    - condicao_spec: dicionário da explicação
    - evidencia: dicionário de evidências do problema
    - prior_alvo: prior marginal do alvo
  - Retorna:
    - float: valor do CBF

In [ ]:
posts_marginais = posteriors(infer_circuito,
                             alvos,
                             evidencia_padrao)
posts_marginais

{'A': <DiscreteFactor representing phi(A:2) at 0x7d9ea6985eb0>,
 'B': <DiscreteFactor representing phi(B:2) at 0x7d9ea76e22a0>,
 'C': <DiscreteFactor representing phi(C:2) at 0x7d9ef0ba1190>,
 'D': <DiscreteFactor representing phi(D:2) at 0x7d9ef0ba1d00>}

Posteriors vs. Priors

In [ ]:
for g in alvos:
    p_prior = priors[g]['defective']
    p_post  = posts_marginais[g].get_value(**{g: 'defective'})

    g_val = gbf(p_post, p_prior)

    print(f"Gate {g:<2} | Prior: {p_prior:.3f} -> Posterior: {p_post:.3f} | GBF: {g_val:>6.2f} [{jeffreys(g_val)}]")

Gate A  | Prior: 0.016 -> Posterior: 0.391 | GBF:  39.45 [Muito forte]
Gate B  | Prior: 0.100 -> Posterior: 0.649 | GBF:  16.65 [Forte]
Gate C  | Prior: 0.150 -> Posterior: 0.446 | GBF:   4.57 [Substancial]
Gate D  | Prior: 0.100 -> Posterior: 0.301 | GBF:   3.88 [Substancial]


Obtendo valores do A sozinho

In [ ]:
p_prior_A = priors['A']['defective']
p_post_A  = posts_marginais['A'].get_value(A='defective')
gbf_A_isolado = gbf(p_post_A, p_prior_A)

print(f'Prior A {p_prior_A:.4f}')
print(f'Posterior A {p_post_A:.4f}')
print(f'GBF do A: {gbf_A_isolado:.4f} [{jeffreys(gbf_A_isolado)}]')

print()

cbf_A_dado_B = cbf_condicional(
    infer=infer_circuito,
    alvo_spec={'A': 'defective'},
    condicao_spec={'B': 'defective'},  # Assumimos B como falho
    evidencia=evidencia_padrao,
    prior_alvo=p_prior_A
)
print(f'Posterior A {p_post_A:.4f}')
print(f"CBF do A (Dado que B falhou): {cbf_A_dado_B:.4f} [{jeffreys(cbf_A_dado_B)}]")

Prior A 0.0160
Posterior A 0.3908
GBF do A: 39.4513 [Muito forte]

Posterior A 0.3908
CBF do A (Dado que B falhou): 4.0359 [Substancial]


# Dominância

- **domina_forte**: dominância forte. $a$ domina fortemente $b$ se $a$ for subconjunto estrito de $b$ e $GBF(a) ≥ GBF(b)$.
  - Parâmetros:
    - a e b: dicionários com valores spec e seus gbf
  - Retorna:
    - bool True ou False
- **domina_fraca**: dominância fraca. $a$ domina fracamente $b$ se $a$ for superconjunto estrito de $b$ ($a$ mais complexo) e $GBF(a) > GBF(b)$
  - Parâmetros:
    - a e b: dicionários com valores spec e seus gbf
  - Retorna:
    - bool True ou False

- **is_minimal**: verifica se uma explicação é minimal, ou seja, se não for dominada forte e fracamente por nenhuma outra explicação.
  - Parâmetros:
    - explicacao: dicionário com valores spec e seus gbf
    - todas: lista dos dicionários com todas explicações
  - Retorna:
    - bool True e False

In [ ]:
from mre import domina_forte, domina_fraca, is_minimal

Explicações hipotéticas

In [ ]:
# Explicação 1: B está defeituoso
prior_1 = priors['B']['defective']
post_1 = infer_circuito.query(['B'],
                                evidence=evidencia_padrao,
                                show_progress=False).get_value(B='defective')
gbf_exp_1 = gbf(post_1, prior_1)

# Explicação 2: B e C estão defeituosos
prior_2 = priors['B']['defective'] * priors['C']['defective']
post_2 = infer_circuito.query(['B', 'C'],
                                  evidence=evidencia_padrao,
                                  show_progress=False).get_value(B='defective', C='defective')
gbf_exp_2 = gbf(post_2, prior_2)

# Explicação 3: B e C estão defeituosos e D está Ok
prior_3 = priors['B']['defective'] * priors['C']['defective'] * priors['D']['ok']
post_3 = infer_circuito.query(['B', 'C', 'D'],
                                 evidence=evidencia_padrao,
                                 show_progress=False).get_value(B='defective', C='defective', D='ok')
gbf_exp_3 = gbf(post_3, prior_3)

Transformando em $a$ e $b$

In [ ]:
exp_1 = {'label': 'B=def',
                  'spec': {'B': 'defective'},
                  'gbf': gbf_exp_1}

exp_2      = {'label': 'B=def, C=def',
                  'spec': {'B': 'defective', 'C': 'defective'},
                  'gbf': gbf_exp_2}

exp_3  = {'label': 'B=def, C=def, D=ok',
                  'spec': {'B': 'defective', 'C': 'defective', 'D': 'ok'},
                  'gbf': gbf_exp_3}

exps = [exp_1, exp_2, exp_3]

Testes de dominância forte

In [ ]:
forte12 = domina_forte(exp_1,exp_2)
print(f'Explicação 1 domina fortemente 2? {forte12}')
print()
forte13 = domina_forte(exp_1, exp_3)
print(f'Explicação 1 domina fortemente 3? {forte13}')
print('------------------------------')
forte21 = domina_forte(exp_2,exp_1)
print(f'Explicação 2 domina fortemente 1? {forte21}')
print()
forte23 = domina_forte(exp_2,exp_3)
print(f'Explicação 2 domina fortemente 3? {forte23}')
print('------------------------------')
forte31 = domina_forte(exp_3, exp_1)
print(f'Explicação 3 domina fortemente 1? {forte31}')
print('------------------------------')
forte32 = domina_forte(exp_3,exp_2)
print(f'Explicação 3 domina fortemente 2? {forte32}')

Explicação 1 domina fortemente 2? False

Explicação 1 domina fortemente 3? False
------------------------------
Explicação 2 domina fortemente 1? False

Explicação 2 domina fortemente 3? True
------------------------------
Explicação 3 domina fortemente 1? False
------------------------------
Explicação 3 domina fortemente 2? False


Testes de dominância fraca

In [ ]:
fraca12 = domina_fraca(exp_1,exp_2)
print(f'Explicação 1 domina fracamente 2? {fraca12}')
print()
fraca13 = domina_fraca(exp_1, exp_3)
print(f'Explicação 1 domina fracamente 3? {fraca13}')
print('------------------------------')
fraca21 = domina_fraca(exp_2,exp_1)
print(f'Explicação 2 domina fracamente 1? {fraca21}')
print()
fraca23 = domina_fraca(exp_2,exp_3)
print(f'Explicação 2 domina fracamente 3? {fraca23}')
print('------------------------------')
fraca31 = domina_fraca(exp_3, exp_1)
print(f'Explicação 3 domina fracamente 1? {fraca31}')
print()
fraca32 = domina_fraca(exp_3,exp_2)
print(f'Explicação 3 domina fracamente 2? {fraca32}')

Explicação 1 domina fracamente 2? False

Explicação 1 domina fracamente 3? False
------------------------------
Explicação 2 domina fracamente 1? True

Explicação 2 domina fracamente 3? False
------------------------------
Explicação 3 domina fracamente 1? True

Explicação 3 domina fracamente 2? False


Minimals

In [ ]:
sobreviventes = []
for exp in exps:
    if is_minimal(exp, exps):
        sobreviventes.append(exp['label'])
        print(f" {exp['label']:<20} -> É uma explicação minimal.")
    else:
        print(f" {exp['label']:<20} -> Foi dominada e descartada.")

 B=def                -> Foi dominada e descartada.
 B=def, C=def         -> É uma explicação minimal.
 B=def, C=def, D=ok   -> Foi dominada e descartada.


# K-MRE

In [ ]:
from mre import todas_instanciacoes, mre, kmre, jeffreys

- **todas_instanciacoes**: calcula GBF de todas as intanciações parciais dos alvos. Para $k$ variáveis-alvo com $n_i$ estados, todas as combinações $2^k - 1$ não vazias de subconjuntos são geradas, e seus estados
  - Parâmetros:
    - infer: VariableElimination do pgmpy
    - alvos: lista das variáveis alvo
    - evidencia: dicionário com evidências observadas
    - estados: dicionário com variaveis e seus estados possíveis
    - priors: dicionário variáveis e probabilidades dos estados
  - Retorna:
    - lista de dicionários, cada um contendo:
      - label: descrição da instanciação
      - spec: variável e seu estado
      - gbf: GBF da instanciação
      - p_post: posterior dada evidência
      - p_prior: priori sem evidência

- **mre**: Most Relevant Explanation. Apresenta a instaciação parcial dos alvos a qual maximiza o GBF.
  - Parâmetros:
    - infer:
    - alvos:
    - evidência
    - estados
    - priors
  - Retorna:
    - dicionário:
      - label: descrição da instanciação
      - spec: variável e seu estado
      - gbf: GBF da instanciação
      - p_post: posterior dada evidência
      - p_prior: priori sem evidência

Calculando todas as combinações possíveis dos 4 (A,B,C,D)

In [ ]:
lista_bruta = todas_instanciacoes(
    infer_circuito,
    alvos,
    evidencia_padrao,
    circuito['estados'],
    priors
)
lista_bruta

[{'label': 'B=defective, C=defective',
  'spec': {'B': 'defective', 'C': 'defective'},
  'gbf': np.float64(42.61521577006454),
  'p_post': np.float64(0.39355813559082226),
  'p_prior': 0.015},
 {'label': 'A=ok, B=defective, C=defective',
  'spec': {'A': 'ok', 'B': 'defective', 'C': 'defective'},
  'gbf': np.float64(42.15465666751823),
  'p_post': np.float64(0.3870761426621533),
  'p_prior': 0.014759999999999999},
 {'label': 'B=defective, C=defective, D=ok',
  'spec': {'B': 'defective', 'C': 'defective', 'D': 'ok'},
  'gbf': np.float64(39.925474436547475),
  'p_post': np.float64(0.3533241943244988),
  'p_prior': 0.0135},
 {'label': 'A=ok, B=defective, C=defective, D=ok',
  'spec': {'A': 'ok', 'B': 'defective', 'C': 'defective', 'D': 'ok'},
  'gbf': np.float64(39.556874119981174),
  'p_post': np.float64(0.34749183902946224),
  'p_prior': 0.013283999999999999},
 {'label': 'A=defective',
  'spec': {'A': 'defective'},
  'gbf': np.float64(39.45128036574133),
  'p_post': np.float64(0.39079524

Combinações geradas pela rede, separando as top 3 por GBF

In [ ]:
for exp in lista_bruta[:3]:
    print(f"GBF: {exp['gbf']:.4f} | {exp['label']}")

GBF: 42.6152 | B=defective, C=defective
GBF: 42.1547 | A=ok, B=defective, C=defective
GBF: 39.9255 | B=defective, C=defective, D=ok


MRE

In [ ]:
melhor_unica = mre(infer_circuito,
                   alvos,
                   evidencia_padrao,
                   circuito['estados'],
                   priors)

print(f"Explicação: {melhor_unica['label']} com GBF {melhor_unica['gbf']:.4f}")

Explicação: B=defective, C=defective com GBF 42.6152


K-MRE

In [ ]:
k_melhores = kmre(
    infer_circuito,
    alvos,
    evidencia_padrao,
    circuito['estados'],
    priors,
    k=3
)
k_melhores

[{'label': 'B=defective, C=defective',
  'spec': {'B': 'defective', 'C': 'defective'},
  'gbf': np.float64(42.61521577006454),
  'p_post': np.float64(0.39355813559082226),
  'p_prior': 0.015},
 {'label': 'A=defective',
  'spec': {'A': 'defective'},
  'gbf': np.float64(39.45128036574133),
  'p_post': np.float64(0.3907952452193906),
  'p_prior': 0.016},
 {'label': 'B=defective, D=defective',
  'spec': {'B': 'defective', 'D': 'defective'},
  'gbf': np.float64(35.88478130050196),
  'p_post': np.float64(0.2660402526846698),
  'p_prior': 0.010000000000000002}]

Exibindo

In [ ]:
for i, exp in enumerate(k_melhores, 1):
    forca = jeffreys(exp['gbf'])
    print(f"{i}º Lugar | GBF: {exp['gbf']:.4f} [{forca:<15}] -> {exp['label']}")

1º Lugar | GBF: 42.6152 [Muito forte    ] -> B=defective, C=defective
2º Lugar | GBF: 39.4513 [Muito forte    ] -> A=defective
3º Lugar | GBF: 35.8848 [Muito forte    ] -> B=defective, D=defective


# Comparativos

**comparar_metodos**: Apresenta outras métricas lado a lado, como K-MRE, K-MAP e K-SIMP.
  - Parâmetros:
    - infer
    - alvos
    - evidencia
    - estados
    - priors
  - Retorna:
    - dicionário com kmre, kmap e ksimp

**imprimir_comparativo**: mostra o comparativo formatado
  - Parâmetros:
    - resultado: saída de comparar_metodos()
    - titulo: opcional
  - Retorna:
    - comparativo formatado

In [ ]:
from mre import comparar_metodos, imprimir_comparativo

Executando K-MRE, K-MAP e K-SIMP

In [ ]:
resultados_finais = comparar_metodos(
    infer_circuito,
    alvos,
    evidencia_padrao,
    circuito['estados'],
    priors,
    k=3
)
resultados_finais

{'kmre': [{'label': 'B=defective, C=defective',
   'spec': {'B': 'defective', 'C': 'defective'},
   'gbf': np.float64(42.61521577006454),
   'p_post': np.float64(0.39355813559082226),
   'p_prior': 0.015},
  {'label': 'A=defective',
   'spec': {'A': 'defective'},
   'gbf': np.float64(39.45128036574133),
   'p_post': np.float64(0.3907952452193906),
   'p_prior': 0.016},
  {'label': 'B=defective, D=defective',
   'spec': {'B': 'defective', 'D': 'defective'},
   'gbf': np.float64(35.88478130050196),
   'p_post': np.float64(0.2660402526846698),
   'p_prior': 0.010000000000000002}],
 'kmap': [{'label': 'A=ok, B=defective, C=defective, D=ok',
   'spec': {'A': 'ok', 'B': 'defective', 'C': 'defective', 'D': 'ok'},
   'p_post': np.float64(0.34749183902946224)},
  {'label': 'A=defective, B=ok, C=ok, D=ok',
   'spec': {'A': 'defective', 'B': 'ok', 'C': 'ok', 'D': 'ok'},
   'p_post': np.float64(0.26837437607187653)},
  {'label': 'A=ok, B=defective, C=ok, D=defective',
   'spec': {'A': 'ok', 'B': '

In [ ]:
imprimir_comparativo(resultados_finais)


  K-MRE    (score = GBF)
  GBF= 42.6152  [Muito forte         ]  B=defective, C=defective
  GBF= 39.4513  [Muito forte         ]  A=defective
  GBF= 35.8848  [Muito forte         ]  B=defective, D=defective

  K-MAP    (score = probabilidade posterior)
  P=   0.3475                          A=ok, B=defective, C=defective, D=ok
  P=   0.2684                          A=defective, B=ok, C=ok, D=ok
  P=   0.2221                          A=ok, B=defective, C=ok, D=defective

  K-SIMP   (score = likelihood)
  L=   0.6989                          D=ok
  L=   0.3011                          D=defective



# Aplicando em exemplos externos

In [ ]:
from pgmpy.example_models import load_model
from pgmpy.inference import VariableElimination
from mre import comparar_metodos, imprimir_comparativo

## Ásia

In [ ]:
modelo_asia = load_model('bnlearn/asia')
infer_asia = VariableElimination(modelo_asia)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
alvos_asia = ['tub', 'lung', 'bronc']

In [ ]:
estados_asia = {}
priors_asia = {}

for alvo in alvos_asia:
    estados_asia[alvo] = modelo_asia.get_cpds(alvo).state_names[alvo]
    q_prior = infer_asia.query([alvo], show_progress=False)
    priors_asia[alvo] = {
        estado: q_prior.get_value(**{alvo: estado})
        for estado in estados_asia[alvo]
    }
    print(f"   Prior de {alvo}: {priors_asia[alvo]}")

   Prior de tub: {'yes': np.float64(0.010400000000000003), 'no': np.float64(0.9896)}
   Prior de lung: {'yes': np.float64(0.055), 'no': np.float64(0.9450000000000001)}
   Prior de bronc: {'yes': np.float64(0.44999999999999996), 'no': np.float64(0.55)}


In [ ]:
evidencia_medica = {
    'xray': 'yes',
    'smoke': 'yes'
}

In [ ]:
resultados_asia = comparar_metodos(
    infer=infer_asia,
    alvos=alvos_asia,
    evidencia=evidencia_medica,
    estados=estados_asia,
    priors=priors_asia,
    k=3
)

imprimir_comparativo(resultados_asia)


  K-MRE    (score = GBF)
  GBF= 31.3532  [Muito forte         ]  lung=yes

  K-MAP    (score = probabilidade posterior)
  P=   0.3836                          bronc=yes, lung=yes, tub=no
  P=   0.2557                          bronc=no, lung=yes, tub=no
  P=   0.1761                          bronc=yes, lung=no, tub=no

  K-SIMP   (score = likelihood)
  L=   0.6000                          bronc=yes
  L=   0.4000                          bronc=no

